# 🤖 ARDY → TRAINROBOT — eigene Text-Bewegungen als Lehrer (ohne eigenes CUDA)

Dieses Notebook erzeugt **Unitree-G1-Bewegungen aus Text** mit **NVIDIA ARDY**
(Autoregressive Diffusion, SIGGRAPH 2026 — `github.com/nv-tlabs/ardy`) und
speichert sie als **G1-QPOS-CSV** — genau das Format, das die App
TRAINROBOT (Testfeld·07) über den Button **„.csv (ARDY)“** als Lehrer importiert.

**Warum dieses Notebook?** ARDY braucht eine CUDA-GPU — du hast keine? Kein Problem:
*Laufzeit → Laufzeittyp ändern → T4 GPU* (kostenlos in Google Colab) und los.

**Ablauf:**
1. GPU prüfen
2. ARDY installieren
3. Hugging-Face-Token eingeben (ARDYs Text-Encoder nutzt das gesperrte Modell
   `meta-llama/Meta-Llama-3-8B-Instruct` — du musst einmalig Zugriff auf die
   Modellseite beantragen und einen Token unter hf.co/settings/tokens erstellen)
4. Bewegungen aus Text erzeugen (Prompts auf **Englisch** funktionieren am besten)
5. CSVs herunterladen → in der App importieren (G1 → GLB-Bewegung → „.csv (ARDY)“)
6. In der App: „Referenz“ antippen → BC vortrainieren → PPO → ⭐ MOTION-KI abspielen

**Gut zu wissen:** Die G1-QPOS-CSV hat 36 Spalten (Wurzel + Quaternion + 29 Gelenke);
das G1-Skelett der App ist **1:1 identisch** — es wird nichts umgerechnet.

## 1) GPU prüfen

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo '❌ Keine GPU — Laufzeit → Laufzeittyp ändern → T4 GPU'

## 2) ARDY installieren (~2–3 Minuten)

In [ ]:
%cd /content
!rm -rf ardy
!git clone https://github.com/nv-tlabs/ardy.git
%cd ardy
# Kern-Inferenz reicht (keine Demo-Extras); baut ein kleines C++-Tool (CMake hat Colab)
!pip install -q -e .
import ardy
print('✅ ARDY installiert')

## 3) Hugging-Face-Token (einmalig)

1. Auf https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct Zugriff beantragen (dauert meist nur Minuten)
2. Token erstellen: https://huggingface.co/settings/tokens (Typ *Read*)
3. Unten einfügen

In [ ]:
from getpass import getpass
import os
token = getpass('HF-Token eingeben (verdeckt): ')
os.environ['HF_TOKEN'] = token
!hf auth login --token "$HF_TOKEN" 2>/dev/null || huggingface-cli login --token "$HF_TOKEN"
print('✅ Token gesetzt')

## 4) Bewegungen aus Text erzeugen

Trage deine Prompts in die Liste ein (Englisch empfohlen). Das Modell
`g1` (25 fps) passt exakt zum App-G1. Pro Prompt entsteht eine CSV in `outputs/`.

In [ ]:
PROMPTS = [
    ('gehen',    'A person walks forward in a straight line.'),
    ('kreuz',    'A person walks in a circle.'),
    ('springen', 'A person jumps.'),
    ('winken',   'A person waves.'),
]
DAUER = 8.0  # Sekunden pro Bewegung

import subprocess
for name, prompt in PROMPTS:
    print(f'▶ {name}: „{prompt}“')
    r = subprocess.run(
        ['python', 'scripts/generate.py', prompt,
         '--model', 'g1', '--duration', str(DAUER), '--output', name],
        capture_output=True, text=True)
    print(r.stdout[-1500:])
    if r.returncode != 0:
        print('⚠️ FEHLER:', r.stderr[-2000:])
!ls -la outputs/

**Wenn der GPU-Speicher reicht?** Der Text-Encoder (Llama-3-8B) ist der größte Brocken.
Auf der kostenlosen T4 (16 GB) hilft bei Speichermangel:
`TEXT_ENCODER_DEVICE=cpu` setzen (ARDY unterstützt CPU-Encodierung — langsamer,
aber die Diffusion läuft weiter auf der GPU) und die Zelle oben erneut ausführen:

In [ ]:
# Bei Bedarf ausführen (Speicher-Fallback):
# import os; os.environ['TEXT_ENCODER_DEVICE'] = 'cpu'

## 5) CSVs herunterladen

In [ ]:
from google.colab import files
import glob, os, zipfile
csvs = glob.glob('outputs/**/*.csv', recursive=True)
assert csvs, 'Keine CSVs gefunden — erst Schritt 4 ausführen'
with zipfile.ZipFile('/content/ardy_g1_motions.zip', 'w') as z:
    for c in csvs:
        z.write(c, arcname=os.path.basename(c))
print('📦', len(csvs), 'CSV(s):', [os.path.basename(c) for c in csvs])
files.download('/content/ardy_g1_motions.zip')

## 6) In der App importieren (Android)

1. ZIP auf dem Handy entpacken (CSV-Dateien liegen los)
2. TRAINROBOT → Roboter **G1** wählen → Bereich **GLB-BEWEGUNG** öffnen
3. **„.csv (ARDY)“** antippen → CSV(s) wählen → sie erscheinen mit dem Zusatz „(ARDY)“
4. **„Referenz“** antippen → Geist zeigt die Bewegung
5. **„BC vortrainieren“** → dann **„Training starten“** (PPO)
6. Danach ⭐ **MOTION-KI**: die trainierte Policy spielt die Bewegung,
   du steuerst bei Bedarf per Joystick/Buttons (Steuern-Mix)

**Fehler beim Import?** Die App sagt dir genau, welche Zeile/Spalte nicht passt —
CSVs müssen von `--model g1` stammen (36 Spalten, kein Header).